<a href="https://colab.research.google.com/github/khalilosss11/Premier-League_2026_Winner_predictor/blob/main/PL_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import argparse
import os
import sys
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import requests
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# config
COMMAND = "both"              # notebook only: "backtest", "live" or "both"

ALL_SEASONS = ["0506", "0607", "0708", "0809", "0910", "1011", "1112", "1213",
               "1314", "1415", "1516", "1617", "1718", "1819", "1920", "2021",
               "2122", "2223", "2324", "2425", "2526"]
FIRST_SEASON = "0506"
MIN_TRAIN_SEASONS = 4         # seasons of history before the first test season
HIST_SEASONS = ALL_SEASONS[ALL_SEASONS.index(FIRST_SEASON):]
LIVE_SEASON = "2627"
DATA_DIR = "data"


TEAM_RENAMES = {"Middlesboro": "Middlesbrough"}

ODDS_SOURCES = ["B365", "Avg", "BbAv", "WH", "IW", "LB", "VC"]

K_ELO, HOME_ADV, REGRESS, ELO_DEFAULT = 20, 60, 0.25, 1450.0
FORM_WINDOW, MKT_WINDOW = 5, 10
FORM_DEFAULT = np.array([1.35, 1.4, 1.4, 4.5, 4.5])   # pts, gf, ga, sot, sot against
MKT_DEFAULT = 1.35

NAMES = ["pts", "gf", "ga", "sot", "sota"]
FEATURES = (["elo_diff"] + [f"h_{n}" for n in NAMES] + [f"a_{n}" for n in NAMES]
            + [f"d_{n}" for n in NAMES] + ["d_mkt"])
LABELS = {"H": 0, "D": 1, "A": 2}


#  data
def load_season(s, refresh=False):
    os.makedirs(DATA_DIR, exist_ok=True)
    path = os.path.join(DATA_DIR, f"E0_{s}.csv")
    if refresh or not os.path.exists(path):
        url = f"https://www.football-data.co.uk/mmz4281/{s}/E0.csv"
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            with open(path, "wb") as f:
                f.write(r.content)
        except requests.RequestException as e:
            if not os.path.exists(path):
                raise
            print(f"Warning: could not refresh {s} ({e}); using cached file.")
    df = pd.read_csv(path, encoding="latin-1")
    df["Season"] = s
    return df


def prepare(df):
    df = df.dropna(subset=["Date", "HomeTeam", "AwayTeam", "FTR"]).copy()
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, format="mixed")
    df["HomeTeam"] = df["HomeTeam"].replace(TEAM_RENAMES)
    df["AwayTeam"] = df["AwayTeam"].replace(TEAM_RENAMES)
    for c in ("HST", "AST"):
        if c not in df:
            df[c] = np.nan
    for o in "HDA":
        cols = [p + o for p in ODDS_SOURCES if p + o in df]
        df["O" + o] = df[cols].bfill(axis=1).iloc[:, 0] if cols else np.nan
    inv = 1 / df[["OH", "OD", "OA"]]                       # margin removed
    df[["pH", "pD", "pA"]] = inv.div(inv.sum(axis=1), axis=0).values
    df["y"] = df["FTR"].map(LABELS)
    return df.sort_values("Date", kind="stable").reset_index(drop=True)


def load_all(include_live=False):
    frames = [load_season(s) for s in HIST_SEASONS]
    if include_live:
        frames.append(load_season(LIVE_SEASON, refresh=True))
    return prepare(pd.concat(frames, ignore_index=True))


def check_names(df):
    """A promoted/relegated swap is 3 teams in, 3 out. Anything else suggests a renamed club."""
    seasons = [s for s in HIST_SEASONS if (df.Season == s).any()]
    teams = {s: set(df.loc[df.Season == s, "HomeTeam"]) | set(df.loc[df.Season == s, "AwayTeam"])
             for s in seasons}
    ok = True
    for prev, cur in zip(seasons, seasons[1:]):
        new, gone = teams[cur] - teams[prev], teams[prev] - teams[cur]
        if len(new) != 3 or len(gone) != 3:
            ok = False
            print(f"Name check {cur}: new={sorted(new)} gone={sorted(gone)} (expected 3 and 3). "
                  "If one is a renamed club, add it to TEAM_RENAMES.")
    if ok:
        print("Team-name check passed.")



class State:
    """Elo + rolling form + market form. Features are read BEFORE update()."""

    def __init__(self):
        self.elo = {}
        self.form = defaultdict(lambda: deque(maxlen=FORM_WINDOW))
        self.mkt = defaultdict(lambda: deque(maxlen=MKT_WINDOW))

    def new_season(self):
        for t in self.elo:
            self.elo[t] = (1 - REGRESS) * self.elo[t] + REGRESS * 1500

    def _form(self, t):
        h = self.form[t]
        return FORM_DEFAULT if not h else np.mean(h, axis=0)

    def _mkt(self, t):
        h = self.mkt[t]
        return MKT_DEFAULT if not h else np.mean(h)

    def features(self, home, away):
        fh, fa = self._form(home), self._form(away)
        elo_d = self.elo.get(home, ELO_DEFAULT) - self.elo.get(away, ELO_DEFAULT)
        return np.concatenate([[elo_d], fh, fa, fh - fa, [self._mkt(home) - self._mkt(away)]])

    def update(self, r):
        h, a = r.HomeTeam, r.AwayTeam
        eh, ea = self.elo.get(h, ELO_DEFAULT), self.elo.get(a, ELO_DEFAULT)
        exp_h = 1 / (1 + 10 ** (-(eh + HOME_ADV - ea) / 400))
        act_h = 1 if r.FTR == "H" else 0.5 if r.FTR == "D" else 0
        d = K_ELO * (act_h - exp_h)
        self.elo[h], self.elo[a] = eh + d, ea - d

        hp, ap = (3, 0) if r.FTR == "H" else (1, 1) if r.FTR == "D" else (0, 3)
        hst = 4.5 if pd.isna(r.HST) else r.HST
        ast = 4.5 if pd.isna(r.AST) else r.AST
        self.form[h].append([hp, r.FTHG, r.FTAG, hst, ast])
        self.form[a].append([ap, r.FTAG, r.FTHG, ast, hst])
        if not pd.isna(r.pH):
            self.mkt[h].append(3 * r.pH + r.pD)
            self.mkt[a].append(3 * r.pA + r.pD)


def replay(df, stop=None, collect=False):
    """Run through the first `stop` matches. Returns the state (and the feature table)."""
    st, rows, prev = State(), [], None
    sub = df.iloc[:stop]
    for r in sub.itertuples():
        if prev is not None and r.Season != prev:
            st.new_season()
        prev = r.Season
        if collect:
            rows.append(st.features(r.HomeTeam, r.AwayTeam))
        st.update(r)
    if collect:
        return st, pd.DataFrame(rows, columns=FEATURES, index=sub.index)
    return st


# models
def make_model():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=2000))


def logloss(P, y):
    return -np.mean(np.log(np.clip(P[np.arange(len(y)), y], 1e-12, 1)))


def test_seasons():
    return HIST_SEASONS[MIN_TRAIN_SEASONS:]


def match_backtest(df, X):
    rows = []
    for s in test_seasons():
        tr, te = df.Season < s, (df.Season == s) & df.pH.notna()
        y_tr, y_te = df.loc[tr, "y"].values, df.loc[te, "y"].values
        freq = np.bincount(y_tr, minlength=3) / len(y_tr)
        model = make_model().fit(X[tr], y_tr)
        rows.append({"season": s,
                     "freq": logloss(np.tile(freq, (te.sum(), 1)), y_te),
                     "bookmakers": logloss(df.loc[te, ["pH", "pD", "pA"]].values, y_te),
                     "model": logloss(model.predict_proba(X[te]), y_te)})
    res = pd.DataFrame(rows).set_index("season")


# title simulation
def points_table(sub):
    pts = defaultdict(int)
    for r in sub.itertuples():
        pts[r.HomeTeam] += 3 if r.FTR == "H" else 1 if r.FTR == "D" else 0
        pts[r.AwayTeam] += 3 if r.FTR == "A" else 1 if r.FTR == "D" else 0
    return pd.Series(pts).sort_values(ascending=False)


def simulate(P, base, hi, ai, n_sims, seed=42):
    """Play the remaining fixtures n_sims times from match probabilities P."""
    rng = np.random.default_rng(seed)
    n = len(base)
    cp = np.cumsum(P, axis=1)[:, :2]
    out = (rng.random((n_sims, len(hi), 1)) > cp[None, :, :]).sum(axis=2)
    hpts = np.where(out == 0, 3, np.where(out == 1, 1, 0))
    apts = np.where(out == 2, 3, np.where(out == 1, 1, 0))
    total = base[None, :] + hpts @ np.eye(n)[hi] + apts @ np.eye(n)[ai]
    tie_broken = total + rng.random(total.shape) * 1e-3
    win = np.bincount(tie_broken.argmax(axis=1), minlength=n) / n_sims
    return win, total.mean(axis=0)


def title_probs(df, X, season, cutoff=None, n_sims=5000):
    """cutoff=None: live mode (unplayed fixtures inferred). cutoff=N: backtest from match N."""
    sea = df[df.Season == season]
    teams = sorted(set(sea.HomeTeam) | set(sea.AwayTeam))
    if cutoff is None:
        played = sea
        done = set(zip(sea.HomeTeam, sea.AwayTeam))
        fixtures = [(h, a) for h in teams for a in teams if h != a and (h, a) not in done]
        stop = sea.index[-1] + 1
    else:
        played, rest = sea.iloc[:cutoff], sea.iloc[cutoff:]
        fixtures = list(zip(rest.HomeTeam, rest.AwayTeam))
        stop = sea.index[0] + cutoff

    tr = df.Season < season                                  # train on the past only
    model = make_model().fit(X[tr], df.loc[tr, "y"])
    state = replay(df, stop)                                 # state frozen at the cutoff
    F = np.array([state.features(h, a) for h, a in fixtures])
    P = model.predict_proba(pd.DataFrame(F, columns=FEATURES))

    ti = {t: i for i, t in enumerate(teams)}
    base = np.zeros(len(teams))
    for t, p in points_table(played).items():
        base[ti[t]] = p
    hi = np.array([ti[h] for h, _ in fixtures])
    ai = np.array([ti[a] for _, a in fixtures])
    win, exp_pts = simulate(P, base, hi, ai, n_sims)
    return pd.DataFrame({"points_now": base, "expected_final_pts": exp_pts.round(1),
                         "title_prob": win}, index=teams).sort_values("title_prob", ascending=False)


def title_backtest(df, X, cutoffs=(50, 100), n_sims=5000):
    summary = []
    for c in cutoffs:
        rows = []
        for s in test_seasons():
            res = title_probs(df, X, s, c, n_sims)
            sea = df[df.Season == s]
            champ = points_table(sea).index[0]
            leader = points_table(sea.iloc[:c]).index[0]
            rows.append({"season": s, "champion": champ, "model_pick": res.index[0],
                         "pick_prob": round(res.title_prob.iloc[0], 2),
                         "prob_on_champ": round(res.title_prob.get(champ, 0.0), 3),
                         "leader": leader})
        t = pd.DataFrame(rows).set_index("season")
        print(f"\nTitle backtest, simulating from match {c}")
        print(t)
        pc = t.prob_on_champ.values
        summary.append({"cutoff": c,
                        "model_pick_hit": (t.model_pick == t.champion).mean(),
                        "leader_hit": (t.leader == t.champion).mean(),
                        "mean_prob_on_champ": pc.mean(),
                        "mean_log_score": np.log(np.clip(pc, 1 / n_sims, 1)).mean()})



def cmd_backtest(args):
    df = load_all(include_live=False)
    check_names(df)
    X = replay(df, collect=True)[1]
    match_backtest(df, X)
    title_backtest(df, X, n_sims=args.sims)


def print_live(df, X, n_sims, top):
    n = int((df.Season == LIVE_SEASON).sum())
    res = title_probs(df, X, LIVE_SEASON, None, n_sims)
    print(f"\nLIVE PREDICTION - season {LIVE_SEASON}: {n} of 380 matches played "
          f"(model trained on seasons {HIST_SEASONS[0]}-{HIST_SEASONS[-1]})")
    print(res.head(top).round(3))

    res.to_csv("live_title_probs.csv")



def cmd_live(args):
    df = load_all(include_live=True)
    check_names(df[df.Season != LIVE_SEASON])
    X = replay(df, collect=True)[1]
    print_live(df, X, args.sims, args.top)


def cmd_both(args):
    df = load_all(include_live=True)          # live rows come last, so history is unaffected
    check_names(df[df.Season != LIVE_SEASON])
    X = replay(df, collect=True)[1]
    match_backtest(df, X)
    title_backtest(df, X, n_sims=args.sims)
    print_live(df, X, max(args.sims, 10000), args.top)


if __name__ == "__main__":
    commands = {"backtest": cmd_backtest, "live": cmd_live, "both": cmd_both}
    if "ipykernel" in sys.modules:
        commands[COMMAND](argparse.Namespace(sims=10000 if COMMAND == "live" else 5000, top=8))
    else:
        ap = argparse.ArgumentParser()
        ap.add_argument("command", choices=list(commands))
        ap.add_argument("--sims", type=int, default=5000)
        ap.add_argument("--top", type=int, default=8)
        args = ap.parse_args()
        commands[args.command](args)

Team-name check passed.

Title backtest, simulating from match 50
          champion model_pick  pick_prob  prob_on_champ      leader
season                                                             
0910       Chelsea    Chelsea       0.55          0.545     Chelsea
1011    Man United    Chelsea       0.63          0.264     Chelsea
1112    Man United   Man City       0.33          0.322  Man United
1213    Man United   Man City       0.39          0.364     Chelsea
1314      Man City   Man City       0.36          0.362     Arsenal
1415       Chelsea   Man City       0.41          0.365     Chelsea
1516     Leicester   Man City       0.74          0.000    Man City
1617       Chelsea   Man City       0.46          0.038    Man City
1718      Man City   Man City       0.54          0.542    Man City
1819      Man City   Man City       0.74          0.741   Liverpool
1920     Liverpool  Liverpool       0.54          0.539   Liverpool
2021      Man City  Liverpool       0.52          